In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5

print("=" * 70)
print("SMART CLASSROOM BEHAVIOUR ANALYSIS")
print("MobileNetV2 Transfer Learning")
print("=" * 70)

print("\nTensorFlow version:", tf.__version__)

# Check GPU
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU detected:", gpus)
else:
    print("WARNING: GPU not detected. Training will be slower.")



dataset_path = (
    "/kaggle/input/datasets/"
    "meetnagadia/human-action-recognition-har-dataset/"
    "Human Action Recognition"
)

train_path = os.path.join(dataset_path, "train")
test_path = os.path.join(dataset_path, "test")

training_csv = os.path.join(
    dataset_path,
    "Training_set.csv"
)

testing_csv = os.path.join(
    dataset_path,
    "Testing_set.csv"
)

print("\nDataset path:")
print(dataset_path)

# Check paths
print("\nChecking dataset...")

print("Training folder:", os.path.exists(train_path))
print("Test folder:", os.path.exists(test_path))
print("Training CSV:", os.path.exists(training_csv))
print("Testing CSV:", os.path.exists(testing_csv))



train_df = pd.read_csv(training_csv)
test_df = pd.read_csv(testing_csv)

print("\nTraining CSV shape:", train_df.shape)
print("Testing CSV shape:", test_df.shape)

print("\nFirst 5 training records:")
print(train_df.head())

print("\nCSV columns:")
print(train_df.columns.tolist())

# Classes
class_list = sorted(train_df["label"].unique())

print("\nNumber of classes:", len(class_list))

print("\nClasses:")
for i, cls in enumerate(class_list):
    print(i, ":", cls)



print("\nCreating image generators...")

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20,

    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_path,

    x_col="filename",
    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="training",

    shuffle=True,
    seed=42
)

validation_generator = validation_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_path,

    x_col="filename",
    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="validation",

    shuffle=False,
    seed=42
)

NUM_CLASSES = len(train_generator.class_indices)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Training images:", train_generator.samples)
print("Validation images:", validation_generator.samples)
print("Number of classes:", NUM_CLASSES)

print("\nClass indices:")
print(train_generator.class_indices)



print("\n" + "=" * 70)
print("BUILDING MOBILENETV2 MODEL")
print("=" * 70)

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze pretrained layers
base_model.trainable = False

# Classification head
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.4)(x)

output = Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

print("\nModel created successfully!")



model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)

print("Model compiled successfully!")



checkpoint = ModelCheckpoint(
    "/kaggle/working/best_smart_classroom_model.keras",

    monitor="val_accuracy",

    save_best_only=True,

    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",

    patience=3,

    restore_best_weights=True,

    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",

    factor=0.2,

    patience=2,

    min_lr=1e-7,

    verbose=1
)



print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

history = model.fit(
    train_generator,

    validation_data=validation_generator,

    epochs=EPOCHS,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)



print("\n" + "=" * 70)
print("SAVING MODEL")
print("=" * 70)

final_model_path = (
    "/kaggle/working/"
    "smart_classroom_final_model.keras"
)

model.save(final_model_path)

print("Model saved successfully:")
print(final_model_path)


train_accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

best_train_accuracy = max(train_accuracy)
best_val_accuracy = max(val_accuracy)

best_epoch = (
    np.argmax(val_accuracy) + 1
)

print("\n" + "=" * 70)
print("TRAINING RESULTS")
print("=" * 70)

print(
    "Best Training Accuracy:",
    round(best_train_accuracy * 100, 2),
    "%"
)

print(
    "Best Validation Accuracy:",
    round(best_val_accuracy * 100, 2),
    "%"
)

print(
    "Best Validation Epoch:",
    best_epoch
)

print(
    "Final Training Accuracy:",
    round(train_accuracy[-1] * 100, 2),
    "%"
)

print(
    "Final Validation Accuracy:",
    round(val_accuracy[-1] * 100, 2),
    "%"
)



plt.figure(figsize=(10, 5))

plt.plot(
    train_accuracy,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    val_accuracy,
    marker="o",
    label="Validation Accuracy"
)

plt.title(
    "MobileNetV2 Training and Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()
plt.grid()

plt.show()



plt.figure(figsize=(10, 5))

plt.plot(
    train_loss,
    marker="o",
    label="Training Loss"
)

plt.plot(
    val_loss,
    marker="o",
    label="Validation Loss"
)

plt.title(
    "MobileNetV2 Training and Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()

plt.show()



print("\n" + "=" * 70)
print("GENERATING VALIDATION PREDICTIONS")
print("=" * 70)

validation_generator.reset()

predictions = model.predict(
    validation_generator,
    verbose=1
)

predicted_classes = np.argmax(
    predictions,
    axis=1
)

true_classes = validation_generator.classes

class_names = list(
    validation_generator.class_indices.keys()
)



print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    true_classes,

    predicted_classes,

    target_names=class_names,

    digits=4
)

print(report)

 

print("\nGenerating confusion matrix...")

cm = confusion_matrix(
    true_classes,
    predicted_classes
)

plt.figure(figsize=(14, 12))

sns.heatmap(
    cm,

    annot=True,

    fmt="d",

    xticklabels=class_names,

    yticklabels=class_names
)

plt.title(
    "MobileNetV2 Confusion Matrix"
)

plt.xlabel(
    "Predicted Behaviour"
)

plt.ylabel(
    "Actual Behaviour"
)

plt.xticks(rotation=45)

plt.yticks(rotation=0)

plt.tight_layout()

plt.show()



print("\n" + "=" * 70)
print("PROJECT COMPLETED")
print("=" * 70)

print("\nDataset:")
print("Training images   :", train_generator.samples)
print("Validation images :", validation_generator.samples)
print("Number of classes :", NUM_CLASSES)

print("\nModel:")
print("Architecture      : MobileNetV2")
print("Input size        : 224 x 224")
print("Transfer learning : Yes")

print("\nPerformance:")
print(
    "Best validation accuracy:",
    round(best_val_accuracy * 100, 2),
    "%"
)

print("\nSaved model:")
print(final_model_path)

print("\n" + "=" * 70)
print("SMART CLASSROOM BEHAVIOUR ANALYSIS FINISHED")
print("=" * 70)

In [ ]:
# ================================================================
# SMART CLASSROOM - MOBILE NET V2 FINE-TUNING
# ================================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

print("=" * 70)
print("SMART CLASSROOM - MOBILE NET V2 FINE-TUNING")
print("=" * 70)


IMG_SIZE = 224
BATCH_SIZE = 32
FINE_TUNE_EPOCHS = 5



dataset_path = (
    "/kaggle/input/datasets/"
    "meetnagadia/human-action-recognition-har-dataset/"
    "Human Action Recognition"
)

train_path = os.path.join(
    dataset_path,
    "train"
)

training_csv = os.path.join(
    dataset_path,
    "Training_set.csv"
)


train_df = pd.read_csv(training_csv)

print("\nTotal images:", len(train_df))
print("Number of classes:", train_df["label"].nunique())


train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20,

    rotation_range=15,
    zoom_range=0.15,

    width_shift_range=0.10,
    height_shift_range=0.10,

    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.20
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,

    directory=train_path,

    x_col="filename",
    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="training",

    shuffle=True,

    seed=42
)

validation_generator = validation_datagen.flow_from_dataframe(
    dataframe=train_df,

    directory=train_path,

    x_col="filename",
    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="validation",

    shuffle=False,

    seed=42
)

print("\nTraining images:",
      train_generator.samples)

print("Validation images:",
      validation_generator.samples)

print("Classes:",
      len(train_generator.class_indices))


model_path = (
    "/kaggle/working/"
    "best_smart_classroom_model.keras"
)

if not os.path.exists(model_path):

    raise FileNotFoundError(
        "Model not found:\n" + model_path
    )

print("\nLoading saved model...")

model = load_model(model_path)

print("Saved model loaded successfully!")



print("\nModel contains",
      len(model.layers),
      "layers.")



for layer in model.layers:

    layer.trainable = False
 


FINE_TUNE_START = 124

for i in range(
    FINE_TUNE_START,
    154
):

    model.layers[i].trainable = True



for i in range(
    FINE_TUNE_START,
    154
):

    if isinstance(
        model.layers[i],
        tf.keras.layers.BatchNormalization
    ):

        model.layers[i].trainable = False



for i in range(
    154,
    len(model.layers)
):

    model.layers[i].trainable = True


print("\n" + "=" * 70)
print("TRAINABLE LAYERS")
print("=" * 70)

trainable_count = 0

for i, layer in enumerate(model.layers):

    if layer.trainable:

        print(
            i,
            layer.name,
            type(layer).__name__
        )

        trainable_count += 1

print(
    "\nTotal trainable layers:",
    trainable_count
)


model.compile(

    optimizer=Adam(
        learning_rate=1e-5
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)

print(
    "\nModel compiled successfully."
)

print(
    "Fine-tuning learning rate: 0.00001"
)


checkpoint = ModelCheckpoint(

    "/kaggle/working/"
    "best_finetuned_smart_classroom_model.keras",

    monitor="val_accuracy",

    save_best_only=True,

    verbose=1
)

early_stopping = EarlyStopping(

    monitor="val_accuracy",

    patience=2,

    restore_best_weights=True,

    verbose=1
)

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.2,

    patience=1,

    min_lr=1e-7,

    verbose=1
)


print("\n" + "=" * 70)
print("STARTING FINE-TUNING")
print("=" * 70)

fine_history = model.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=FINE_TUNE_EPOCHS,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ]
)


final_model_path = (
    "/kaggle/working/"
    "smart_classroom_finetuned_final.keras"
)

model.save(final_model_path)

print("\nFinal fine-tuned model saved:")
print(final_model_path)


train_acc = fine_history.history["accuracy"]

val_acc = fine_history.history["val_accuracy"]

train_loss = fine_history.history["loss"]

val_loss = fine_history.history["val_loss"]

best_train_accuracy = max(train_acc)

best_val_accuracy = max(val_acc)

best_epoch = (
    np.argmax(val_acc) + 1
)

print("\n" + "=" * 70)
print("FINE-TUNING RESULTS")
print("=" * 70)

print(
    "Best Training Accuracy:",
    round(
        best_train_accuracy * 100,
        2
    ),
    "%"
)

print(
    "Best Validation Accuracy:",
    round(
        best_val_accuracy * 100,
        2
    ),
    "%"
)

print(
    "Best Epoch:",
    best_epoch
)

print(
    "Final Validation Accuracy:",
    round(
        val_acc[-1] * 100,
        2
    ),
    "%"
)


original_accuracy = 62.58

new_accuracy = (
    best_val_accuracy * 100
)

improvement = (
    new_accuracy -
    original_accuracy
)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    "Original Validation Accuracy:",
    original_accuracy,
    "%"
)

print(
    "Fine-Tuned Validation Accuracy:",
    round(
        new_accuracy,
        2
    ),
    "%"
)

print(
    "Improvement:",
    round(
        improvement,
        2
    ),
    "percentage points"
)

if improvement > 0:

    print(
        "\nSUCCESS! Fine-tuning improved the model."
    )

elif improvement == 0:

    print(
        "\nFine-tuning produced the same accuracy."
    )

else:

    print(
        "\nFine-tuning did not improve the original model."
    )


plt.figure(figsize=(10, 5))

plt.plot(
    train_acc,
    marker="o",
    label="Fine-Tuning Training Accuracy"
)

plt.plot(
    val_acc,
    marker="o",
    label="Fine-Tuning Validation Accuracy"
)

plt.title(
    "MobileNetV2 Fine-Tuning Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid()

plt.show()


plt.figure(figsize=(10, 5))

plt.plot(
    train_loss,
    marker="o",
    label="Fine-Tuning Training Loss"
)

plt.plot(
    val_loss,
    marker="o",
    label="Fine-Tuning Validation Loss"
)

plt.title(
    "MobileNetV2 Fine-Tuning Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid()

plt.show()

print("\n" + "=" * 70)
print("FINE-TUNING COMPLETED")
print("=" * 70)

In [ ]:
# Save the current fine-tuned model
model.save("/kaggle/working/smart_classroom_finetuned_final.keras")

print(" Model saved successfully!")
print("/kaggle/working/smart_classroom_finetuned_final.keras")

In [ ]:
model.save("/kaggle/working/best_finetuned_smart_classroom_model.keras")

print(" Fine-tuned model saved successfully!")

In [ ]:
model.save("/kaggle/working/smart_classroom_finetuned_final.keras")

print(" Final model saved successfully!")